# Late-Interaction Retrieval, by Hand

A toy re-implementation of the ColBERT-style **MaxSim** scoring behind Perplexity's
`pplx-embed-v2-late` (Oct 2026): instead of one vector per document, keep one vector
*per token* and score a query as

$$\mathrm{MaxSim}(Q, D) = \sum_{q \in Q} \max_{d \in D} \cos(q, d)$$

each query token finds its best-matching document token. We build a tiny synthetic
corpus, then show (1) MaxSim beating mean-pooled single-vector retrieval on a
distractor query, (2) the shared-space trick: query vectors from a smaller model
(a noisy projection) still retrieve the right doc.

In [ ]:
import numpy as np
rng = np.random.default_rng(42)
D = 128  # token vector dim, same as pplx-embed-v2-late


def token_vec(seed_vec, noise=0.05):
    v = seed_vec + rng.normal(0, noise, D)
    return v / np.linalg.norm(v)


def maxsim(query, doc):
    """query, doc: (n_q, D), (n_d, D) unit vectors -> MaxSim score."""
    sims = query @ doc.T          # cosine, both unit norm
    return sims.max(axis=1).sum()


def dense_score(query, doc):
    """Single-vector baseline: mean pool both sides, cosine."""
    return (query.mean(0) @ doc.mean(0))


# --- synthetic 'concepts': well-separated anchors in R^128
anchors = {k: rng.normal(0, 1, D) / np.linalg.norm(rng.normal(0, 1, D)) for k in
           ['refund', 'invoice', 'shipped', 'tracking', 'pizza', 'hiking', 'invoice_id']}

# Doc A: invoice/refund vocabulary. Doc B: pizza/hiking (irrelevant) + filler.
def doc(tokens, per_token=6):
    return np.stack([token_vec(anchors[t]) for t in tokens for _ in range(per_token)])

docA = doc(['refund', 'invoice', 'invoice_id'])
docB = doc(['pizza', 'hiking', 'pizza', 'hiking'])
query = doc(['invoice', 'refund'])[:12]           # 12 query tokens

print('doc shapes:', docA.shape, docB.shape, query.shape)

In [ ]:
sA = maxsim(query, docA)
sB = maxsim(query, docB)
dA = dense_score(query, docA)
dB = dense_score(query, docB)
print(f'MaxSim:  docA={sA:.2f}  docB={sB:.2f}  -> winner: {"docA" if sA > sB else "docB"}')
print(f'Dense:   docA={dA:.3f}  docB={dB:.3f}  -> winner: {"docA" if dA > dB else "docB"}')

# MaxSim: every query token latches onto its best doc token, so the relevant
# doc wins decisively. Mean pooling dilutes the signal into one soup vector.
assert sA > sB, 'MaxSim must rank the relevant doc first'
assert (sA - sB) > (dA - dB) * 20, 'MaxSim margin should dwarf the dense margin'
print('asserts pass: MaxSim ranks relevant doc with a much bigger margin')

## The shared-space trick

`pplx-embed-v2-late` lets the 0.6B model query a 9B-built index because both embed
into one space. Emulate a small-model query as the true query embedding plus
noise, then measure whether top-1 retrieval survives.

In [ ]:
def small_model_query(q, noise=0.25, seed=0):
    r = np.random.default_rng(seed)
    noisy = q + r.normal(0, noise, q.shape)
    return noisy / np.linalg.norm(noisy, axis=1, keepdims=True)

corpus = {'docA (invoice)': docA, 'docB (pizza)': docB,
          'docC (invoice+shipped)': doc(['invoice', 'shipped'])}

trials, kept = 50, 0
for t in range(trials):
    q_small = small_model_query(query, seed=t)
    scores = {name: maxsim(q_small, d) for name, d in corpus.items()}
    if max(scores, key=scores.get) == 'docA (invoice)':
        kept += 1
print(f'noisy 0.6B-style queries keep top-1 on the 9B index: {kept}/{trials}')
assert kept >= 45, 'cross-model querying must mostly preserve top-1'
print('assert passes: shared space survives a noisy small-model query')

In [ ]:
# Storage check behind d4 in the post: late-interaction vs dense, per 1k-token doc.
late_fp16_kb = 128 * 1000 * 2 / 1024
dense_4096_kb = 4096 * 4 / 1024
print(f'late-interaction fp16: {late_fp16_kb:.0f} KB/doc | dense fp32 4096-d: {dense_4096_kb:.0f} KB/doc')
assert abs(late_fp16_kb - 250) < 1 and abs(dense_4096_kb - 16) < 1
print('assert passes: matches the d4 diagram numbers')